# UD4.05 — Visualizar para evaluar: mirar si un modelo funciona

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
UD4 — Visualización de datos · 14 horas

Criterio 2.e · Material de partida de la práctica P4.2

## De qué va este cuaderno

La UD3 terminó con `X` e `y` construidos, la partición hecha y un punto de referencia
medido. Dijo explícitamente que entrenar era la UD5, y lo sigue siendo.

Aquí se resuelve el paso que queda en medio y que casi nadie enseña: **alguien te da
las predicciones de un modelo y tienes que decidir si funciona**. Ese es el criterio
2.e —*se han evaluado los resultados obtenidos*— y es un problema de visualización,
no de estadística, por una razón concreta:

> Un número resume. Y todo resumen tiene una forma de fallar que consiste
> precisamente en que el resumen sale bien.

Una exactitud del 82 % puede ser un modelo excelente, un modelo que no ha aprendido
nada, o un modelo que funciona para tres cuartas partes de tus clientes y falla con la
otra. **Los tres casos dan el mismo número.** Distinguirlos es lo que hacen los seis
gráficos de este cuaderno.

### Lo que no se usa aquí

Nada de `scikit-learn`. Todas las curvas se construyen con NumPy y unas pocas líneas,
igual que en el cuaderno `UD3_05`, y por el mismo motivo: una curva ROC que has
ordenado y acumulado tú no vuelve a ser una caja negra nunca más. En la UD5 se
llamará a `sklearn.metrics` y se sabrá qué hace por dentro.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(20262027)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
plt.rcParams["figure.dpi"] = 110

print("numpy", np.__version__, "· pandas", pd.__version__)

In [ ]:
# Solo en Google Colab: descarga los ficheros de datos de la unidad.
import os
import urllib.request

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD4/datos/")
FICHEROS = ["predicciones_abandono.csv", "predicciones_gasto.csv",
            "curvas_aprendizaje.csv"]

os.makedirs("datos", exist_ok=True)
for nombre in FICHEROS:
    destino = os.path.join("datos", nombre)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(BASE + nombre, destino)
        print("descargado:", nombre)
    else:
        print("ya está:", nombre)

## 1. Los datos: el modelo de la UD3, ya entrenado

El caso es el mismo de `UD3_05`: **qué clientes de la tienda van a dejar de comprar**,
para poder hacerles una oferta antes de que se vayan.

| Columna | Es |
|---|---|
| `cliente` | El identificador |
| `segmento` | `reciente`, `consolidado` o `veterano`, por los días de historial |
| `recencia_dias`, `frecuencia`, `antiguedad_dias`, `monetario` | Cuatro de las características de la UD3 |
| `abandona` | **La verdad.** 1 si no hizo ningún pedido en los 180 días siguientes al corte |
| `puntuacion` | Lo que dice el modelo A: un número entre 0 y 1 |
| `puntuacion_modelo_b` | Lo que dice el modelo B, un candidato alternativo |

Ojo a la última fila: hay **dos** modelos que hay que comparar. Guárdalo para la
sección 6, que es donde se decide.

In [ ]:
abandono = pd.read_csv(os.path.join("datos", "predicciones_abandono.csv"))

verdad = abandono["abandona"].to_numpy()
modelo_a = abandono["puntuacion"].to_numpy()
modelo_b = abandono["puntuacion_modelo_b"].to_numpy()

positivos = int(verdad.sum())
negativos = len(verdad) - positivos

print(f"{len(abandono)} clientes")
print(f"  se van   (clase 1): {positivos:>4}  ({positivos / len(verdad) * 100:.1f} %)")
print(f"  se quedan (clase 0): {negativos:>4}  "
      f"({negativos / len(verdad) * 100:.1f} %)")
print()
print("Reparto por segmento:")
print(abandono.groupby("segmento", observed=True)
      .agg(clientes=("cliente", "size"),
           tasa_abandono=("abandona", "mean")).round(3).to_string())
print()
abandono.head()

### 1.1 El primer número, y por qué no vale

Empecemos como empieza todo el mundo: umbral en 0,5 y a ver la exactitud.

In [ ]:
prediccion = (modelo_a >= 0.5).astype(int)
exactitud = float((prediccion == verdad).mean())

# El modelo tonto: contestar siempre la clase mayoritaria.
exactitud_tonta = float(max(verdad.mean(), 1 - verdad.mean()))

print(f"Exactitud del modelo con umbral 0,5:       {exactitud:.3f}")
print(f"Exactitud de contestar siempre 'se queda': {exactitud_tonta:.3f}")
print(f"Diferencia:                                {exactitud - exactitud_tonta:+.3f}")
print()
print(f"El modelo saca un {exactitud * 100:.0f} %, que suena bien, y una regla que no "
      f"mira los datos saca")
print(f"un {exactitud_tonta * 100:.0f} %. Toda la aportación del modelo cabe en esos "
      f"{(exactitud - exactitud_tonta) * 100:.0f} puntos.")
print()
print("Con clases desequilibradas la exactitud es casi siempre un número inútil, y")
print("cuanto más desequilibradas, más inútil. Si solo el 2 % de los clientes se")
print("fuera, contestar 'se queda' siempre acertaría el 98 % de las veces.")
print()
print("Así que hay que mirar otra cosa. Vamos por orden.")

## 2. Cuatro modelos con la misma exactitud

Esta sección es el argumento de todo el cuaderno. Cuatro modelos sobre los mismos 400
clientes, **los cuatro con exactamente la misma exactitud**, y cuatro comportamientos
que no se parecen en nada.

Las matrices están escritas a mano para que los cuatro casos sean exactos.

In [ ]:
# Cada matriz es [[VN, FP], [FN, VP]]: filas la verdad, columnas la predicción.
casos = {
    "A · equilibrado":      np.array([[268,  36], [ 36,  60]]),
    "B · muy prudente":     np.array([[301,   3], [ 69,  27]]),
    "C · muy alarmista":    np.array([[236,  68], [  4,  92]]),
    "D · no ha aprendido":  np.array([[304,   0], [ 72,  24]]),
}


def metricas(matriz):
    """Las cinco cifras que se sacan de una matriz de confusión 2x2."""
    (vn, fp), (fn, vp) = matriz
    total = vn + fp + fn + vp
    precision = vp / (vp + fp) if vp + fp else 0.0
    exhaustividad = vp / (vp + fn) if vp + fn else 0.0
    f1 = (2 * precision * exhaustividad / (precision + exhaustividad)
          if precision + exhaustividad else 0.0)
    return {"exactitud": (vp + vn) / total, "precision": precision,
            "exhaustividad": exhaustividad, "f1": f1,
            "marcados": vp + fp}


print(f"{'modelo':>22} {'exactitud':>10} {'precisión':>10} "
      f"{'exhaustiv.':>11} {'F1':>7} {'marcados':>9}")
print("-" * 76)
for nombre, matriz in casos.items():
    m = metricas(matriz)
    print(f"{nombre:>22} {m['exactitud']:>10.3f} {m['precision']:>10.3f} "
          f"{m['exhaustividad']:>11.3f} {m['f1']:>7.3f} {m['marcados']:>9}")

print()
print("Cuatro exactitudes idénticas: 0,820. Y mira las otras cuatro columnas.")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
fig.suptitle("Cuatro modelos con la MISMA exactitud (0,820)", fontsize=15,
             fontweight="bold")

etiquetas = ["se queda", "se va"]
for ax, (nombre, matriz) in zip(axes, casos.items()):
    imagen = ax.imshow(matriz, cmap="Blues", vmin=0, vmax=310)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{matriz[i, j]}", ha="center", va="center",
                    fontsize=15, fontweight="bold",
                    color="white" if matriz[i, j] > 155 else "black")
    m = metricas(matriz)
    ax.set_title(f"{nombre}\nprecisión {m['precision']:.2f} · "
                 f"exhaustividad {m['exhaustividad']:.2f}",
                 fontweight="bold", fontsize=10)
    ax.set_xticks([0, 1], [f"dice\n«{e}»" for e in etiquetas], fontsize=8)
    ax.set_yticks([0, 1], [f"es\n«{e}»" for e in etiquetas], fontsize=8)

fig.tight_layout()
plt.show()

print("Qué hace cada uno, en una frase:")
print()
print("A  Se equivoca por igual en las dos direcciones. Es el que la mayoría")
print("   imagina cuando le dicen '82 % de exactitud'.")
print()
print("B  Solo avisa cuando está segurísimo: 30 avisos, 27 aciertos. Pero deja")
print("   escapar a 69 de los 96 clientes que se iban. Si el aviso cuesta caro")
print("   (una llamada comercial), es el bueno.")
print()
print("C  Avisa de casi todos: caza 92 de los 96. A cambio, de cada 160 avisos, 68")
print("   son falsas alarmas. Si perder un cliente cuesta mucho más que un correo")
print("   automático, es el bueno.")
print()
print("D  NO HA APRENDIDO NADA de la clase que importa: acierta 24 de 96 y NUNCA")
print("   se equivoca al decir 'se va'. Mira su matriz: la columna de falsos")
print("   positivos es cero porque casi nunca dice que alguien se va.")
print()
print("Los cuatro son 0,820. La exactitud no distingue entre B, C y D, y la")
print("diferencia entre elegir B o C son decenas de miles de euros.")

## 3. La matriz de confusión, y las dos formas de normalizarla

La matriz de confusión es el primer gráfico. Se construye con `pd.crosstab`, que
cuenta las combinaciones de dos columnas, y se dibuja con `imshow`.

Y hay una decisión que cambia la conclusión: **normalizar por filas o por columnas**.

| Normalización | Cada celda es | Contesta a |
|---|---|---|
| **Por filas** (la verdad) | «de los que **eran** de esta clase, qué proporción...» | Exhaustividad. *¿A cuántos de los que se iban los cacé?* |
| **Por columnas** (la predicción) | «de los que **dije** de esta clase, qué proporción...» | Precisión. *De los que avisé, ¿cuántos se iban de verdad?* |

Son dos preguntas distintas y las dos importan, pero **importan a personas
distintas**: la primera al que quiere retener clientes, la segunda al que paga las
llamadas.

In [ ]:
matriz = pd.crosstab(abandono["abandona"], prediccion,
                     rownames=["verdad"], colnames=["predicción"])
print("La matriz de confusión del modelo A con umbral 0,5:")
print()
print(matriz.to_string())
print()

por_filas = matriz.div(matriz.sum(axis=1), axis=0)
por_columnas = matriz.div(matriz.sum(axis=0), axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
fig.suptitle("La misma matriz, tres lecturas", fontsize=15, fontweight="bold")

vistas = [
    (matriz.to_numpy(), "Recuentos", "d", "Blues", None),
    (por_filas.to_numpy(), "Normalizada por FILAS\n"
     "«de los que eran así, qué proporción...» → exhaustividad", ".1%",
     "Greens", (0, 1)),
    (por_columnas.to_numpy(), "Normalizada por COLUMNAS\n"
     "«de los que dije así, qué proporción...» → precisión", ".1%",
     "Oranges", (0, 1)),
]

for ax, (datos, titulo, formato, mapa, limites) in zip(axes, vistas):
    minimo, maximo = limites if limites else (None, None)
    imagen = ax.imshow(datos, cmap=mapa, vmin=minimo, vmax=maximo)
    umbral_texto = (datos.max() * 0.55 if limites is None else 0.55)
    for i in range(2):
        for j in range(2):
            texto = (f"{datos[i, j]:.0f}" if formato == "d"
                     else f"{datos[i, j]:.1%}")
            ax.text(j, i, texto, ha="center", va="center", fontsize=14,
                    fontweight="bold",
                    color="white" if datos[i, j] > umbral_texto else "black")
    ax.set_title(titulo, fontweight="bold", fontsize=10)
    ax.set_xticks([0, 1], ["dice\n«se queda»", "dice\n«se va»"], fontsize=9)
    ax.set_yticks([0, 1], ["es\n«se queda»", "es\n«se va»"], fontsize=9)

fig.tight_layout()
plt.show()

(vn, fp), (fn, vp) = matriz.to_numpy()
print(f"Los cuatro números, con su nombre y su significado en este caso:")
print()
print(f"  Verdaderos negativos  {vn:>4}   clientes que se quedan y el modelo deja en paz")
print(f"  Falsos positivos      {fp:>4}   clientes que se quedan y reciben una oferta "
      f"que no hacía falta")
print(f"  Falsos negativos      {fn:>4}   clientes que se van y NADIE los llama")
print(f"  Verdaderos positivos  {vp:>4}   clientes que se van y el modelo detecta")
print()
print(f"Exhaustividad (fila de abajo): {vp / (vp + fn):.1%} de los que se iban")
print(f"Precisión (columna derecha):   {vp / (vp + fp):.1%} de los avisos eran ciertos")
print()
print("Las dos cifras son del MISMO modelo con el MISMO umbral. Enseñar la primera")
print("y no la segunda es lo que hace que un modelo parezca mejor de lo que es.")

## 4. El reparto de las puntuaciones, que es el gráfico que lo explica todo

La matriz de confusión no es lo que el modelo produce: es lo que queda **después de
aplicar un umbral**. Lo que el modelo produce es un número continuo, y el gráfico que
hay que mirar es el reparto de ese número **separado por la clase verdadera**.

Se lee así:

- **Las dos distribuciones separadas** → el modelo distingue las clases.
- **Las dos distribuciones encima** → el modelo no distingue nada, y ningún umbral lo
  va a arreglar.
- **La zona donde se solapan** → los clientes sobre los que el modelo duda, que son
  los que el umbral reparte a un lado o a otro.

La consecuencia práctica: **el umbral no cambia el modelo, cambia dónde se corta**.
Mover el umbral solo redistribuye los errores del solape.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("El reparto de las puntuaciones del modelo A, por clase verdadera",
             fontsize=15, fontweight="bold")

bordes = np.linspace(0, 1, 31)

ax = axes[0, 0]
ax.hist(modelo_a[verdad == 0], bins=bordes, alpha=0.65, label="se queda (0)",
        color="#2471a3")
ax.hist(modelo_a[verdad == 1], bins=bordes, alpha=0.65, label="se va (1)",
        color="#c0392b")
ax.axvline(0.5, color="black", linestyle="--", linewidth=2, label="umbral 0,5")
ax.set_title("Recuentos: la clase mayoritaria tapa a la otra", fontweight="bold",
             fontsize=10)
ax.set_xlabel("Puntuación del modelo")
ax.set_ylabel("Clientes")
ax.legend(fontsize=8)

# Normalizado: cada clase suma 1, y entonces se comparan las FORMAS.
ax = axes[0, 1]
for clase, color, etiqueta in ((0, "#2471a3", "se queda (0)"),
                               (1, "#c0392b", "se va (1)")):
    puntos = modelo_a[verdad == clase]
    pesos = np.ones(len(puntos)) / len(puntos)
    ax.hist(puntos, bins=bordes, weights=pesos, alpha=0.6, label=etiqueta,
            color=color)
ax.axvline(0.5, color="black", linestyle="--", linewidth=2)
ax.set_title("Normalizado por clase: ahora se comparan las formas",
             fontweight="bold", fontsize=10)
ax.set_xlabel("Puntuación del modelo")
ax.set_ylabel("Proporción de su clase")
ax.legend(fontsize=8)

# La función de distribución acumulada, que no depende de ningún `bins`.
ax = axes[1, 0]
for clase, color, etiqueta in ((0, "#2471a3", "se queda (0)"),
                               (1, "#c0392b", "se va (1)")):
    puntos = np.sort(modelo_a[verdad == clase])
    ax.plot(puntos, np.arange(1, len(puntos) + 1) / len(puntos),
            linewidth=2.2, color=color, label=etiqueta)
ax.axvline(0.5, color="black", linestyle="--", linewidth=2)
ax.set_title("Distribución acumulada: sin parámetros que ajustar",
             fontweight="bold", fontsize=10)
ax.set_xlabel("Puntuación del modelo")
ax.set_ylabel("Proporción acumulada")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Qué pasa al mover el umbral.
ax = axes[1, 1]
umbrales = np.linspace(0.02, 0.98, 200)
exhaustividades = [(modelo_a[verdad == 1] >= u).mean() for u in umbrales]
precisiones = [((verdad[modelo_a >= u] == 1).mean()
                if (modelo_a >= u).any() else np.nan) for u in umbrales]
proporcion_marcada = [(modelo_a >= u).mean() for u in umbrales]

ax.plot(umbrales, exhaustividades, linewidth=2.4, color="#c0392b",
        label="Exhaustividad")
ax.plot(umbrales, precisiones, linewidth=2.4, color="#1d6b3f", label="Precisión")
ax.plot(umbrales, proporcion_marcada, linewidth=1.8, color="0.45",
        linestyle=":", label="Proporción de clientes marcados")
ax.axvline(0.5, color="black", linestyle="--", linewidth=2)
ax.set_title("Precisión y exhaustividad según el umbral\n"
             "Una sube cuando la otra baja: siempre", fontweight="bold", fontsize=10)
ax.set_xlabel("Umbral")
ax.set_ylabel("Proporción")
ax.set_ylim(0, 1.02)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

solape_bajo = float(np.percentile(modelo_a[verdad == 1], 10))
solape_alto = float(np.percentile(modelo_a[verdad == 0], 90))
print(f"El 10 % de los que SE VAN puntúan por debajo de {solape_bajo:.3f}.")
print(f"El 10 % de los que SE QUEDAN puntúan por encima de {solape_alto:.3f}.")
print()
print(f"Entre {solape_bajo:.3f} y {solape_alto:.3f} las dos clases se mezclan, y ahí")
print("hay clientes de los dos tipos con la misma puntuación. Ningún umbral los")
print("separa, porque el modelo no los distingue.")
print()
print("Eso es el techo del modelo, y se ve en el primer gráfico. Si al mirarlo las")
print("dos distribuciones estuvieran una encima de la otra, la conclusión sería")
print("'este modelo no sirve' y no habría que probar veinte umbrales para saberlo.")

## 5. El umbral sale del coste, no de 0,5

El 0,5 no tiene nada de especial: es el valor que sale por defecto porque hay que
poner alguno. El umbral correcto **sale de lo que cuesta cada tipo de error**, y esa
es la última fila de la ficha del modelo de la UD3, la que casi nunca se escribe.

En este caso los costes son estos, y son inventados pero razonables:

| Error | Cuesta | Por qué |
|---|---|---|
| **Falso positivo** | 20 € | Se le hace un descuento a alguien que se iba a quedar |
| **Falso negativo** | 180 € | Se pierde un cliente que se podría haber retenido |

Con un falso negativo nueve veces más caro que un falso positivo, el umbral óptimo
**no puede** ser 0,5. Y se puede calcular.

In [ ]:
COSTE_FP = 20.0
COSTE_FN = 180.0

umbrales = np.linspace(0.01, 0.99, 300)
coste_total = []
for u in umbrales:
    pred = modelo_a >= u
    falsos_positivos = int((pred & (verdad == 0)).sum())
    falsos_negativos = int((~pred & (verdad == 1)).sum())
    coste_total.append(falsos_positivos * COSTE_FP + falsos_negativos * COSTE_FN)
coste_total = np.array(coste_total)

umbral_optimo = float(umbrales[int(np.argmin(coste_total))])
coste_optimo = float(coste_total.min())
coste_en_medio = float(coste_total[int(np.argmin(np.abs(umbrales - 0.5)))])
coste_sin_modelo = positivos * COSTE_FN          # no hacer nada
coste_llamar_a_todos = negativos * COSTE_FP      # dar el descuento a todo el mundo

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 5.2))

izq.plot(umbrales, coste_total, linewidth=2.4, color="#1a5276")
izq.axvline(umbral_optimo, color="#1d6b3f", linestyle="--", linewidth=2.2,
            label=f"óptimo: {umbral_optimo:.2f} → {coste_optimo:,.0f} €")
izq.axvline(0.5, color="#c0392b", linestyle=":", linewidth=2.2,
            label=f"por defecto 0,50 → {coste_en_medio:,.0f} €")
izq.axhline(coste_sin_modelo, color="0.5", linestyle="-.", linewidth=1.6,
            label=f"sin modelo → {coste_sin_modelo:,.0f} €")
izq.plot(umbral_optimo, coste_optimo, "*", color="#1d6b3f", markersize=18,
         markeredgecolor="white", markeredgewidth=1.2)
izq.set_title("Coste esperado según el umbral\n"
              f"falso positivo {COSTE_FP:.0f} € · falso negativo {COSTE_FN:.0f} €",
              fontweight="bold", fontsize=11)
izq.set_xlabel("Umbral")
izq.set_ylabel("Coste total (€)")
izq.legend(fontsize=8, loc="upper left")
izq.grid(True, alpha=0.3)

# Cómo se mueve el óptimo si cambia la proporción de costes.
proporciones = np.array([1, 2, 3, 5, 9, 15, 30])
optimos = []
for proporcion in proporciones:
    costes = [(int(((modelo_a >= u) & (verdad == 0)).sum())
               + proporcion * int(((modelo_a < u) & (verdad == 1)).sum()))
              for u in umbrales]
    optimos.append(umbrales[int(np.argmin(costes))])

der.plot(proporciones, optimos, "o-", linewidth=2.4, markersize=8, color="#6c3483")
der.axhline(0.5, color="#c0392b", linestyle=":", linewidth=2,
            label="el 0,5 por defecto")
der.axvline(9, color="#1d6b3f", linestyle="--", linewidth=1.8,
            label="nuestro caso: 180/20 = 9")
der.set_xscale("log")
der.set_xticks(proporciones, [str(p) for p in proporciones])
der.set_title("Dónde está el umbral óptimo según lo caro que sea\n"
              "equivocarse en la clase positiva", fontweight="bold", fontsize=11)
der.set_xlabel("Cuántas veces más cuesta un falso negativo que un falso positivo")
der.set_ylabel("Umbral óptimo")
der.set_ylim(0, 1)
der.legend(fontsize=8)
der.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

pred_optima = modelo_a >= umbral_optimo
print(f"Umbral óptimo: {umbral_optimo:.3f}, no 0,5.")
print()
print(f"{'estrategia':>34} {'coste':>12} {'ahorro':>12}")
print("-" * 60)
for nombre, coste in [("no hacer nada", coste_sin_modelo),
                      ("descuento a todo el mundo", coste_llamar_a_todos),
                      ("modelo con umbral 0,50", coste_en_medio),
                      ("modelo con umbral óptimo", coste_optimo)]:
    print(f"{nombre:>34} {coste:>10,.0f} € "
          f"{coste_sin_modelo - coste:>10,.0f} €")

print()
print(f"Elegir bien el umbral vale {coste_en_medio - coste_optimo:,.0f} € sobre el")
print(f"mismo modelo, sin reentrenar nada. Es el {(coste_en_medio - coste_optimo) / coste_en_medio * 100:.0f} %")
print("del coste que tenía con el umbral por defecto.")
print()
print("Y la lectura del gráfico de la derecha: el umbral óptimo baja al aumentar el")
print("coste del falso negativo, y tiene sentido. Si perder un cliente es carísimo,")
print("hay que avisar ante la menor sospecha, y eso es un umbral bajo.")
print()
print("Nada de esto se puede decidir mirando la exactitud, porque la exactitud")
print("cuenta los dos errores como si costaran lo mismo. Y no cuestan lo mismo casi")
print("nunca.")

## 6. Las dos curvas, construidas a mano

Un umbral da un punto. **Las curvas recorren todos los umbrales de golpe**, y por eso
describen el modelo entero en lugar de una configuración concreta.

### Cómo se construyen, que son cinco líneas

1. Ordenar los clientes **de mayor a menor puntuación**.
2. Recorrerlos en ese orden: bajar el umbral es ir marcando uno más.
3. En cada paso, acumular aciertos y falsas alarmas: `np.cumsum`.
4. Dividir por el total de cada clase.
5. Dibujar.

Eso es todo. Una curva ROC no es más que un `argsort` y dos `cumsum`.

In [ ]:
def curvas(verdad, puntuacion):
    """Devuelve las dos curvas de evaluación y sus áreas, sin scikit-learn.

    Se ordena por puntuación descendente. Recorrer ese orden equivale a bajar el
    umbral cliente a cliente: en el paso k se han marcado los k de mayor puntuación.

    Devuelve un diccionario con:
      fpr, tpr      la curva ROC (falsas alarmas frente a aciertos)
      exhaustividad, precision   la curva de precisión-exhaustividad
      auc           área bajo la ROC: probabilidad de ordenar bien un par al azar
      ap            precisión media: el área bajo la curva de precisión-exhaustividad
    """
    orden = np.argsort(-puntuacion)
    v = verdad[orden]

    aciertos = np.cumsum(v)              # verdaderos positivos acumulados
    falsas = np.cumsum(1 - v)            # falsos positivos acumulados
    total_positivos, total_negativos = v.sum(), len(v) - v.sum()

    tpr = aciertos / total_positivos     # exhaustividad
    fpr = falsas / total_negativos       # proporción de falsas alarmas
    precision = aciertos / (aciertos + falsas)

    # Áreas. La de la ROC por trapecios; la precisión media como la suma de la
    # precisión en cada punto por lo que aumenta la exhaustividad en ese paso, que es
    # la definición estándar y no interpola.
    auc = float(np.trapezoid(np.r_[0, tpr], np.r_[0, fpr]))
    ap = float(np.sum(np.diff(np.r_[0, tpr]) * precision))
    return {"fpr": fpr, "tpr": tpr, "exhaustividad": tpr, "precision": precision,
            "auc": auc, "ap": ap}


curva_a = curvas(verdad, modelo_a)
curva_b = curvas(verdad, modelo_b)

# La regla tonta de la UD3, para tener con qué comparar: cuanto mayor la recencia,
# mayor el riesgo. Una sola columna y ningún modelo.
curva_referencia = curvas(verdad, abandono["recencia_dias"].to_numpy().astype(float))

print(f"{'':>26} {'AUC-ROC':>9} {'precisión media':>17}")
print("-" * 56)
print(f"{'Modelo A':>26} {curva_a['auc']:>9.4f} {curva_a['ap']:>17.4f}")
print(f"{'Modelo B':>26} {curva_b['auc']:>9.4f} {curva_b['ap']:>17.4f}")
print(f"{'Punto de referencia UD3':>26} {curva_referencia['auc']:>9.4f} "
      f"{curva_referencia['ap']:>17.4f}")
print(f"{'Azar':>26} {0.5:>9.4f} {verdad.mean():>17.4f}")
print()
print("Dos cosas que hay que leer aquí, y las dos importan.")
print()
print(f"PRIMERA. El modelo saca {curva_a['auc']:.3f} de AUC y ORDENAR POR LA RECENCIA")
print(f"A SECAS saca {curva_referencia['auc']:.3f}. El modelo gana, pero por "
      f"{(curva_a['auc'] - curva_referencia['auc']) * 1000:.0f} milésimas.")
print("Ocho características, un ajuste y una tubería de datos, para mejorar eso.")
print()
print("Eso es un hallazgo y hay que informar de él, no esconderlo. Puede que el")
print("modelo no merezca el coste de mantenerlo, y esa conversación no se puede")
print("tener sin el punto de referencia medido. Es justo lo que la UD3 dejó dicho:")
print("sin punto de referencia no se sabe si un modelo aporta algo.")
print()
print("SEGUNDA. Los modelos A y B tienen el MISMO AUC hasta el cuarto decimal, y la")
print("misma precisión media. Según estas dos cifras son el mismo modelo. En la")
print("sección 7 se verá que no lo son en absoluto.")

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Las dos curvas de evaluación de un clasificador", fontsize=15,
             fontweight="bold")

# --- ROC ---
izq.plot(curva_a["fpr"], curva_a["tpr"], linewidth=2.6, color="#1a5276",
         label=f"Modelo A · AUC = {curva_a['auc']:.3f}")
izq.plot(curva_referencia["fpr"], curva_referencia["tpr"], linewidth=2,
         color="#e67e22", linestyle="-.",
         label=f"Recencia sola (UD3) · AUC = {curva_referencia['auc']:.3f}")
izq.plot([0, 1], [0, 1], "k--", linewidth=1.4, label="Azar · AUC = 0,500")
izq.fill_between(curva_a["fpr"], curva_a["tpr"], curva_a["fpr"], alpha=0.12,
                 color="#1a5276")

# El punto de trabajo elegido en la sección 5.
fp_optimo = ((modelo_a >= umbral_optimo) & (verdad == 0)).sum() / negativos
tp_optimo = ((modelo_a >= umbral_optimo) & (verdad == 1)).sum() / positivos
izq.plot(fp_optimo, tp_optimo, "*", color="#1d6b3f", markersize=20,
         markeredgecolor="white", markeredgewidth=1.4,
         label=f"Punto de trabajo (umbral {umbral_optimo:.2f})")
izq.annotate(f"Aquí caza el {tp_optimo:.0%} de los que se van\n"
             f"a cambio de {fp_optimo:.0%} de falsas alarmas",
             xy=(fp_optimo, tp_optimo), xytext=(0.34, 0.42),
             arrowprops=dict(arrowstyle="->", color="#1d6b3f", lw=1.8),
             fontsize=9, color="#145a32", fontweight="bold",
             bbox=dict(boxstyle="round", facecolor="#eafaf1", alpha=0.9))

izq.set_title("Curva ROC\nEl área es la probabilidad de ordenar bien\n"
              "un par (uno que se va, uno que se queda)",
              fontweight="bold", fontsize=11)
izq.set_xlabel("Proporción de falsas alarmas (1 − especificidad)")
izq.set_ylabel("Exhaustividad (proporción de positivos cazados)")
izq.legend(fontsize=8, loc="lower right")
izq.grid(True, alpha=0.3)
izq.set_xlim(0, 1)
izq.set_ylim(0, 1.02)

# --- Precisión-exhaustividad ---
der.plot(curva_a["exhaustividad"], curva_a["precision"], linewidth=2.6,
         color="#1a5276", label=f"Modelo A · precisión media = {curva_a['ap']:.3f}")
der.plot(curva_referencia["exhaustividad"], curva_referencia["precision"],
         linewidth=2, color="#e67e22", linestyle="-.",
         label=f"Recencia sola · {curva_referencia['ap']:.3f}")
der.axhline(verdad.mean(), color="black", linestyle="--", linewidth=1.4,
            label=f"Azar = prevalencia = {verdad.mean():.3f}")

precision_optima = (verdad[modelo_a >= umbral_optimo] == 1).mean()
der.plot(tp_optimo, precision_optima, "*", color="#1d6b3f", markersize=20,
         markeredgecolor="white", markeredgewidth=1.4,
         label=f"Punto de trabajo (umbral {umbral_optimo:.2f})")

der.set_title("Curva de precisión-exhaustividad\n"
              "La que hay que mirar con clases desequilibradas",
              fontweight="bold", fontsize=11)
der.set_xlabel("Exhaustividad")
der.set_ylabel("Precisión")
der.legend(fontsize=8, loc="upper right")
der.grid(True, alpha=0.3)
der.set_xlim(0, 1)
der.set_ylim(0, 1.02)

fig.tight_layout()
plt.show()

print("Cómo se leen las dos, y por qué hacen falta las dos:")
print()
print("ROC. El eje X son las falsas alarmas sobre el total de NEGATIVOS, que aquí son")
print(f"     {negativos}. Como los negativos son muchos, el denominador es grande y la")
print("     curva sube deprisa. Es la razón por la que la ROC parece optimista con")
print("     clases desequilibradas: unas pocas falsas alarmas sobre trescientos")
print("     negativos apenas mueven el eje X.")
print()
print("PRECISIÓN-EXHAUSTIVIDAD. El eje Y es la precisión, cuyo denominador son los")
print("     clientes MARCADOS. Ahí las falsas alarmas sí pesan, porque compiten con")
print(f"     los {positivos} positivos que hay. Esta curva no perdona.")
print()
print(f"     Su línea de azar no es una diagonal: es la horizontal en "
      f"{verdad.mean():.3f}, la")
print("     prevalencia. Un clasificador que conteste al azar acierta esa proporción")
print("     de las veces que dice 'se va', haga lo que haga.")
print()
print("Regla: si la clase que te interesa es la minoritaria —fraude, avería, abandono,")
print("enfermedad—, la curva que decide es la de precisión-exhaustividad.")

## 7. Dos modelos con la misma curva ROC

Aquí está la parte que hay que entender de este cuaderno.

En la sección 6 los modelos A y B salieron con **el mismo AUC hasta el cuarto
decimal**. No es casualidad ni suerte: el modelo B es una **transformación monótona**
del A. Se obtiene elevando la puntuación de A a una potencia menor que uno, y como esa
operación conserva el orden, los dos modelos ordenan a los 400 clientes exactamente
igual.

Y la curva ROC **solo depende del orden**. Por eso son idénticas.

La pregunta, entonces: si ordenan igual, ¿son el mismo modelo? Depende de qué se vaya
a hacer con la puntuación.

In [ ]:
print("Los dos modelos ordenan igual a los clientes:")
orden_a = np.argsort(-modelo_a)
orden_b = np.argsort(-modelo_b)
print(f"  ¿El orden es el mismo?  {np.array_equal(orden_a, orden_b)}")
print(f"  AUC de A: {curva_a['auc']:.6f}")
print(f"  AUC de B: {curva_b['auc']:.6f}")
print()
print("Y no dicen lo mismo:")
print()
print(f"{'cliente':>10} {'verdad':>7} {'modelo A':>10} {'modelo B':>10}")
print("-" * 42)
for i in orden_a[[0, 40, 120, 250, 399]]:
    print(f"{abandono['cliente'][i]:>10} {verdad[i]:>7} "
          f"{modelo_a[i]:>10.3f} {modelo_b[i]:>10.3f}")
print()
print(f"Puntuación media: modelo A {modelo_a.mean():.3f}, "
      f"modelo B {modelo_b.mean():.3f}")
print(f"Proporción real de abandono: {verdad.mean():.3f}")
print()
print("El modelo A tiene una puntuación media que coincide con la proporción real de")
print("abandono. El modelo B la duplica. Los dos ordenan igual y solo uno de los dos")
print("números se puede leer como una probabilidad.")

In [ ]:
def fiabilidad(verdad, puntuacion, n_tramos=8):
    """Diagrama de fiabilidad: predicción media frente a frecuencia observada.

    Se parten las puntuaciones en tramos y en cada uno se comparan dos cosas:
      - lo que el modelo dijo de media,
      - la proporción de positivos que hubo de verdad.

    Si el modelo está bien calibrado, los dos números coinciden y los puntos caen
    sobre la diagonal. Es el único gráfico que detecta este problema.
    """
    bordes = np.linspace(0, 1, n_tramos + 1)
    tramo = np.clip(np.digitize(puntuacion, bordes) - 1, 0, n_tramos - 1)
    filas = []
    for k in range(n_tramos):
        dentro = tramo == k
        if dentro.sum() >= 5:            # tramos con menos de 5 casos no dicen nada
            filas.append((float(puntuacion[dentro].mean()),
                          float(verdad[dentro].mean()),
                          int(dentro.sum())))
    return np.array(filas)


fiab_a = fiabilidad(verdad, modelo_a)
fiab_b = fiabilidad(verdad, modelo_b)

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.4))
fig.suptitle("Dos modelos con la MISMA curva ROC y distinta calibración",
             fontsize=15, fontweight="bold")

# 1. Las dos ROC, superpuestas: son la misma línea.
ax = axes[0]
ax.plot(curva_a["fpr"], curva_a["tpr"], linewidth=5, color="#1a5276", alpha=0.45,
        label=f"Modelo A · AUC {curva_a['auc']:.4f}")
ax.plot(curva_b["fpr"], curva_b["tpr"], linewidth=1.8, color="#c0392b",
        linestyle="--", label=f"Modelo B · AUC {curva_b['auc']:.4f}")
ax.plot([0, 1], [0, 1], "k:", linewidth=1.2)
ax.set_title("Las curvas ROC\nSon la misma línea, exactamente",
             fontweight="bold", fontsize=11)
ax.set_xlabel("Falsas alarmas")
ax.set_ylabel("Exhaustividad")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3)

# 2. El diagrama de fiabilidad: aquí se separan.
ax = axes[1]
ax.plot([0, 1], [0, 1], "k--", linewidth=1.6, label="Calibración perfecta")
ax.plot(fiab_a[:, 0], fiab_a[:, 1], "o-", linewidth=2.4, markersize=9,
        color="#1a5276", label="Modelo A")
ax.plot(fiab_b[:, 0], fiab_b[:, 1], "s-", linewidth=2.4, markersize=9,
        color="#c0392b", label="Modelo B")
for pred, real, n in fiab_b:
    ax.annotate(f"n={n}", (pred, real), textcoords="offset points",
                xytext=(6, -12), fontsize=7, color="#c0392b")
ax.fill_between([0, 1], [0, 1], [0, 0], alpha=0.06, color="red")
ax.text(0.62, 0.22, "Zona de\nSOBRECONFIANZA\n(dice más de lo que hay)",
        fontsize=8.5, color="#922b21", ha="center", fontweight="bold")
ax.set_title("Diagrama de fiabilidad\nEl único gráfico que los distingue",
             fontweight="bold", fontsize=11)
ax.set_xlabel("Puntuación media que da el modelo")
ax.set_ylabel("Proporción real de abandono")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)

# 3. Qué cuesta en euros la mala calibración.
ax = axes[2]
VALOR_CLIENTE = 400.0
COSTE_DESCUENTO = 20.0
tramos = np.linspace(0.05, 0.95, 19)
for puntuacion, nombre, color in ((modelo_a, "Modelo A", "#1a5276"),
                                  (modelo_b, "Modelo B", "#c0392b")):
    # Decisión: dar el descuento si el beneficio esperado es positivo, usando la
    # probabilidad que dice el modelo. Se compara con lo que pasa de verdad.
    esperado, real = [], []
    for u in tramos:
        marcados = puntuacion >= u
        if marcados.sum() < 5:
            continue
        beneficio_esperado = (puntuacion[marcados].mean() * VALOR_CLIENTE
                              - COSTE_DESCUENTO) * marcados.sum()
        beneficio_real = (verdad[marcados].mean() * VALOR_CLIENTE
                          - COSTE_DESCUENTO) * marcados.sum()
        esperado.append(beneficio_esperado)
        real.append(beneficio_real)
    ax.plot(esperado, real, "o-", color=color, linewidth=2.2, markersize=6,
            label=nombre)

limite = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, limite], [0, limite], "k--", linewidth=1.5,
        label="Lo prometido = lo obtenido")
ax.set_title("Lo que el modelo promete frente a lo que da\n"
             f"(cliente retenido {VALOR_CLIENTE:.0f} € · "
             f"descuento {COSTE_DESCUENTO:.0f} €)",
             fontweight="bold", fontsize=11)
ax.set_xlabel("Beneficio que el modelo dice que habrá (€)")
ax.set_ylabel("Beneficio que hay de verdad (€)")
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

error_calibracion_a = float(np.average(np.abs(fiab_a[:, 0] - fiab_a[:, 1]),
                                       weights=fiab_a[:, 2]))
error_calibracion_b = float(np.average(np.abs(fiab_b[:, 0] - fiab_b[:, 1]),
                                       weights=fiab_b[:, 2]))

print(f"Error de calibración medio (ponderado por el tamaño del tramo):")
print(f"  Modelo A: {error_calibracion_a:.3f}")
print(f"  Modelo B: {error_calibracion_b:.3f}")
print()
print("La conclusión, que es la de esta sección entera:")
print()
print("SI LA DECISIÓN ES UN ORDEN —a quién llamo primero, qué diez casos reviso hoy,")
print("cómo ordeno la lista de resultados— los dos modelos son EXACTAMENTE iguales,")
print("porque ordenan igual. La ROC es la métrica correcta y no hay más que hablar.")
print()
tramo_medio = fiab_b[np.argmin(np.abs(fiab_b[:, 0] - 0.45))]
print("SI LA DECISIÓN USA EL NÚMERO —doy el descuento si la probabilidad por el valor")
print("del cliente supera el coste— entonces el modelo B miente. En su tramo de")
print(f"puntuación {tramo_medio[0]:.2f} la frecuencia real de abandono es "
      f"{tramo_medio[1]:.2f}, y con")
print("ese número se aprueban gastos que no se recuperan. Se ve en el tercer panel:")
print("la línea del modelo B está muy por debajo de la diagonal, o sea que promete")
print("mucho más de lo que da.")
print()
print(f"Y la comprobación más rápida de todas: la puntuación media del modelo A es")
print(f"{modelo_a.mean():.3f} y la prevalencia real es {verdad.mean():.3f}. "
      f"La del modelo B es {modelo_b.mean():.3f}.")
print("Si la media de las probabilidades que da un modelo no se parece a la")
print("proporción real de positivos, el modelo no está calibrado y ya está.")
print()
print("Y NADA de esto aparece en el AUC, ni en la exactitud, ni en el F1, ni en la")
print("matriz de confusión. Solo en el diagrama de fiabilidad.")

## 8. Por segmentos: el modelo que funciona de media

Todo lo anterior se ha calculado sobre los 400 clientes juntos. Un modelo puede tener
un AUC decente en el conjunto y **no funcionar en absoluto para un grupo concreto**.

Eso no es un detalle técnico. Es un problema de dos clases:

- **Práctico:** si el modelo no sirve para los clientes nuevos, la campaña de
  captación se hace a ciegas y nadie se ha enterado.
- **De equidad:** cuando el segmento es una característica de las personas —edad,
  sexo, procedencia, código postal— un modelo que funciona de media y falla en un
  grupo es un modelo que discrimina. Y el número global lo tapa.

La comprobación es siempre la misma: **desglosar todas las métricas por grupo**. Y
hay que desglosar también **el punto de referencia**, porque la pregunta no es «¿va
bien el modelo en este grupo?» sino «¿aporta el modelo algo en este grupo?», que es
distinta y a veces tiene respuesta contraria.

In [ ]:
segmentos = ["reciente", "consolidado", "veterano"]
recencia = abandono["recencia_dias"].to_numpy().astype(float)
resumen = []
for segmento in segmentos:
    dentro = (abandono["segmento"] == segmento).to_numpy()
    v, p = verdad[dentro], modelo_a[dentro]
    marcados = p >= umbral_optimo
    resumen.append({
        "segmento": segmento,
        "clientes": int(dentro.sum()),
        "tasa_real": float(v.mean()),
        "auc": curvas(v, p)["auc"],
        # El mismo AUC de la regla tonta de la UD3, dentro de este segmento. Sin esta
        # columna no se puede saber si el modelo aporta algo AQUÍ.
        "auc_referencia": curvas(v, recencia[dentro])["auc"],
        "marcados": int(marcados.sum()),
        "precision": float(v[marcados].mean()) if marcados.any() else np.nan,
        "exhaustividad": float(v[marcados].sum() / v.sum()) if v.sum() else np.nan,
        "exactitud": float((marcados.astype(int) == v).mean()),
    })
resumen = pd.DataFrame(resumen).set_index("segmento")
resumen["gana_al_referente"] = resumen["auc"] > resumen["auc_referencia"]

print(f"Métricas globales del modelo A con umbral {umbral_optimo:.2f}:")
marcados_global = modelo_a >= umbral_optimo
print(f"  AUC {curva_a['auc']:.3f} · "
      f"precisión {verdad[marcados_global].mean():.3f} · "
      f"exhaustividad {verdad[marcados_global].sum() / positivos:.3f} · "
      f"exactitud {(marcados_global.astype(int) == verdad).mean():.3f}")
print()
print("Y desglosadas por segmento:")
print()
print(resumen.round(3).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.2))
fig.suptitle("El mismo modelo, desglosado por segmento", fontsize=15,
             fontweight="bold")

colores = {"reciente": "#c0392b", "consolidado": "#e67e22", "veterano": "#1d6b3f"}

# 1. Las tres curvas ROC.
ax = axes[0]
for segmento in segmentos:
    dentro = (abandono["segmento"] == segmento).to_numpy()
    curva = curvas(verdad[dentro], modelo_a[dentro])
    ax.plot(curva["fpr"], curva["tpr"], linewidth=2.4, color=colores[segmento],
            label=f"{segmento} · AUC {curva['auc']:.3f}")
ax.plot(curva_a["fpr"], curva_a["tpr"], linewidth=3.4, color="black", alpha=0.35,
        label=f"TODOS · AUC {curva_a['auc']:.3f}")
ax.plot([0, 1], [0, 1], "k:", linewidth=1.2, label="Azar")
ax.set_title("Curva ROC por segmento", fontweight="bold", fontsize=11)
ax.set_xlabel("Falsas alarmas")
ax.set_ylabel("Exhaustividad")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3)

# 2. El modelo frente a la regla tonta, dentro de cada segmento. Es el panel que
# convierte el desglose en una decisión.
ax = axes[1]
posicion = np.arange(len(segmentos))
ancho = 0.36
ax.bar(posicion - ancho / 2, resumen["auc"], ancho, label="Modelo",
       color="#2471a3", edgecolor="black", linewidth=0.6)
ax.bar(posicion + ancho / 2, resumen["auc_referencia"], ancho,
       label="Recencia sola (UD3)", color="#e67e22", edgecolor="black",
       linewidth=0.6)
ax.axhline(0.5, color="0.4", linestyle=":", linewidth=1.6, label="Azar")
for i, fila in enumerate(resumen.itertuples()):
    gana = fila.auc > fila.auc_referencia
    ax.annotate("gana" if gana else "PIERDE",
                (i, max(fila.auc, fila.auc_referencia) + 0.012),
                ha="center", fontsize=9, fontweight="bold",
                color="#1d6b3f" if gana else "#922b21")
ax.set_xticks(posicion, segmentos)
ax.set_ylabel("AUC")
ax.set_ylim(0.4, 0.95)
ax.set_title("¿Aporta el modelo algo en cada segmento?\nAUC del modelo frente al "
             "punto de referencia", fontweight="bold", fontsize=11)
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3, axis="y")

# 3. El reparto de puntuaciones por clase, dentro de cada segmento. Es donde se ve
# POR QUÉ falla: en el segmento reciente las dos clases están encima.
ax = axes[2]
for i, segmento in enumerate(segmentos):
    dentro = (abandono["segmento"] == segmento).to_numpy()
    for clase, desplazamiento, marca in ((0, -0.16, "o"), (1, 0.16, "s")):
        puntos = modelo_a[dentro & (verdad == clase)]
        ax.scatter(np.full(len(puntos), i + desplazamiento)
                   + rng.normal(0, 0.035, len(puntos)),
                   puntos, s=13, alpha=0.55, marker=marca,
                   color="#2471a3" if clase == 0 else "#c0392b",
                   label=(("se queda (0)", "se va (1)")[clase] if i == 0 else None))
        ax.plot([i + desplazamiento - 0.09, i + desplazamiento + 0.09],
                [puntos.mean()] * 2, color="black", linewidth=2.4)
ax.axhline(umbral_optimo, color="#1d6b3f", linestyle="--", linewidth=1.8,
           label=f"umbral {umbral_optimo:.2f}")
ax.set_xticks(range(len(segmentos)), segmentos)
ax.set_ylabel("Puntuación del modelo")
ax.set_title("Puntuaciones por clase dentro de cada segmento\n"
             "Las rayas negras son las medias", fontweight="bold", fontsize=11)
ax.legend(fontsize=8, loc="upper left")
ax.grid(True, alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

peor = resumen["auc"].idxmin()
mejor = resumen["auc"].idxmax()
print(f"El AUC global es {curva_a['auc']:.3f}, y por segmento va de "
      f"{resumen['auc'].min():.3f} ({peor}) a {resumen['auc'].max():.3f} ({mejor}).")
print()
print("Y ahora el hallazgo, que no se ve en ninguna cifra global:")
print()
for fila in resumen.itertuples():
    veredicto = ("gana al punto de referencia" if fila.gana_al_referente
                 else "PIERDE contra el punto de referencia")
    print(f"  {fila.Index:12s}  modelo {fila.auc:.3f}  "
          f"recencia sola {fila.auc_referencia:.3f}   {veredicto}")
print()
print(f"En «{peor}» el modelo saca {resumen.loc[peor, 'auc']:.3f} y ordenar por la "
      f"recencia a secas saca")
print(f"{resumen.loc[peor, 'auc_referencia']:.3f}. Ahí el modelo no solo aporta poco: "
      f"aporta MENOS QUE NADA,")
print(f"porque una regla de una línea lo hace mejor. Y son "
      f"{resumen.loc[peor, 'clientes']} clientes, la cuarta")
print("parte del total.")
print()
print("La explicación de negocio está en el nombre del segmento: son clientes con")
print("poco historial, y las características del modelo —recencia, frecuencia,")
print("antigüedad, gasto acumulado— necesitan historial para decir algo. Con dos")
print("pedidos no dicen, y al combinarlas el modelo mete ruido de siete columnas")
print("que en ese grupo no informan.")
print()
print("Fíjate también en la precisión de la tabla, en el punto de trabajo:")
for fila in resumen.itertuples():
    print(f"  {fila.Index:12s}  tasa real de abandono {fila.tasa_real:.3f}  "
          f"precisión {fila.precision:.3f}")
print()
print("La precisión va de "
      f"{resumen['precision'].min():.2f} a {resumen['precision'].max():.2f} "
      "con el MISMO umbral, y el orden es el de la")
print("tasa real de abandono de cada grupo. Eso no es un fallo del modelo: es")
print("aritmética. Con el mismo umbral, un grupo donde se va menos gente produce")
print("más falsas alarmas, siempre.")
print()
print("Tiene una consecuencia que se olvida: **un umbral único reparte la carga de")
print("error de forma desigual entre los grupos**. Si el grupo fuera una")
print("característica de las personas y no la antigüedad como cliente, eso sería")
print("directamente un problema de equidad, y el número global lo tapa.")
print()
print("Qué se hace con esto. Tres opciones, y ninguna es 'nada':")
print("  1. Declararlo y actuar: el modelo se usa en consolidados y veteranos, y en")
print("     los recientes se usa la regla de la recencia, que ahí funciona mejor.")
print("  2. Un modelo aparte para los recientes, con otras características.")
print("  3. Conseguir características que sí funcionen con poco historial: de dónde")
print("     vino el cliente, qué compró primero, si abrió los correos.")
print()
print("La primera es gratis y se puede hacer esta tarde. Y no se le habría ocurrido")
print(f"a nadie sin este desglose, porque el AUC global del modelo "
      f"({curva_a['auc']:.3f}) es")
print(f"mejor que el de la referencia ({curva_referencia['auc']:.3f}), y con solo esas "
      f"dos cifras la conclusión")
print("perezosa era «usar el modelo, y punto».")

## 9. Regresión: los residuos

Cuando lo que se predice es un número y no una clase, la matriz de confusión no
aplica y el gráfico que manda es otro: **el residuo frente al valor predicho**.

El residuo es el error de cada observación: `predicho − real`. Y el gráfico se lee
buscando **estructura**, porque un modelo que ha aprovechado toda la información deja
residuos sin estructura:

| Lo que se ve | Significa |
|---|---|
| Nube sin forma centrada en cero | Bien: no queda información en el error |
| Un embudo que se abre | El error crece con el valor. El modelo es fiable abajo y no arriba |
| Una curva | Queda relación sin capturar. Falta una transformación o un término |
| Desplazada del cero | Sesgo sistemático: se equivoca siempre en el mismo sentido |

El segundo caso tiene nombre, **heterocedasticidad**, y es el más frecuente y el que
ningún número global delata.

In [ ]:
gasto = pd.read_csv(os.path.join("datos", "predicciones_gasto.csv"))
real = gasto["gasto_real"].to_numpy()
predicho = gasto["gasto_predicho"].to_numpy()
residuo = predicho - real

# Las cifras globales que se informan siempre.
error_medio = float(residuo.mean())
error_absoluto_medio = float(np.abs(residuo).mean())
raiz_error_cuadratico = float(np.sqrt((residuo ** 2).mean()))
r2 = 1 - float((residuo ** 2).sum() / ((real - real.mean()) ** 2).sum())

print(f"Las cifras que se informan de un modelo de regresión:")
print(f"  R²                              {r2:>10.3f}")
print(f"  Error absoluto medio (MAE)      {error_absoluto_medio:>10.2f} €")
print(f"  Raíz del error cuadrático (RMSE){raiz_error_cuadratico:>10.2f} €")
print(f"  Error medio (sesgo)             {error_medio:>+10.2f} €")
print()
print(f"Un R² de {r2:.2f} explica el {r2 * 100:.0f} % de la varianza. Suena a modelo")
print("razonable, y hay dos problemas graves que ese número no menciona.")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9.5))
fig.suptitle("Los seis gráficos de un modelo de regresión", fontsize=15,
             fontweight="bold")

# 1. Predicho frente a real, con la diagonal.
ax = axes[0, 0]
ax.scatter(real, predicho, s=14, alpha=0.5, edgecolors="none", color="#2471a3")
tope = max(real.max(), predicho.max()) * 1.03
ax.plot([0, tope], [0, tope], "k--", linewidth=1.6, label="Predicción perfecta")
ax.set_title(f"Predicho frente a real · R² = {r2:.3f}", fontweight="bold",
             fontsize=10)
ax.set_xlabel("Gasto real (€)")
ax.set_ylabel("Gasto predicho (€)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. EL gráfico: residuo frente a predicho.
ax = axes[0, 1]
ax.scatter(predicho, residuo, s=14, alpha=0.5, edgecolors="none", color="#c0392b")
ax.axhline(0, color="black", linewidth=1.6)
# La banda de la desviación del residuo por tramos, que es lo que hace visible el
# embudo: si fuera constante, las dos líneas serían horizontales.
bordes = np.quantile(predicho, np.linspace(0, 1, 11))
centros = (bordes[:-1] + bordes[1:]) / 2
desviaciones = np.array([residuo[(predicho >= a) & (predicho < b)].std()
                         for a, b in zip(bordes[:-1], bordes[1:])])
ax.plot(centros, desviaciones, color="#1d6b3f", linewidth=2.2,
        label="+1 desviación del residuo")
ax.plot(centros, -desviaciones, color="#1d6b3f", linewidth=2.2)
ax.set_title("Residuo frente a predicho\nEL EMBUDO: el error crece con el valor",
             fontweight="bold", fontsize=10, color="#922b21")
ax.set_xlabel("Gasto predicho (€)")
ax.set_ylabel("Residuo: predicho − real (€)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. El residuo relativo, que es el que importa cuando el error es proporcional.
ax = axes[0, 2]
relativo = residuo / np.maximum(real, 1) * 100
ax.scatter(predicho, relativo, s=14, alpha=0.5, edgecolors="none", color="#6c3483")
ax.axhline(0, color="black", linewidth=1.6)
ax.set_ylim(-100, 100)
ax.set_title("Residuo RELATIVO (%)\nAquí el embudo desaparece: el error es "
             "proporcional", fontweight="bold", fontsize=10)
ax.set_xlabel("Gasto predicho (€)")
ax.set_ylabel("Error relativo (%)")
ax.grid(True, alpha=0.3)

# 4. El histograma del residuo.
ax = axes[1, 0]
ax.hist(residuo, bins=45, color="#5dade2", edgecolor="#1a5276", alpha=0.85)
ax.axvline(0, color="black", linewidth=1.8, label="cero")
ax.axvline(error_medio, color="#c0392b", linestyle="--", linewidth=2,
           label=f"media = {error_medio:+.1f} €")
ax.set_title("Reparto del residuo\nDesplazado a la izquierda: sesgo",
             fontweight="bold", fontsize=10)
ax.set_xlabel("Residuo (€)")
ax.set_ylabel("Clientes")
ax.legend(fontsize=8)

# 5. El error por tramos de valor real, que es lo que revela dónde falla.
ax = axes[1, 1]
tramos = pd.qcut(real, 6, duplicates="drop")
por_tramo = pd.DataFrame({"real": real, "residuo": residuo,
                          "tramo": tramos}).groupby("tramo", observed=True).agg(
    n=("residuo", "size"), sesgo=("residuo", "mean"),
    error_absoluto=("residuo", lambda s: np.abs(s).mean()))
posicion = np.arange(len(por_tramo))
ax.bar(posicion, por_tramo["sesgo"], color=np.where(por_tramo["sesgo"] < 0,
                                                    "#c0392b", "#1d6b3f"),
       edgecolor="black", linewidth=0.6)
ax.axhline(0, color="black", linewidth=1.2)
ax.set_xticks(posicion, [f"{i.left:,.0f}\n{i.right:,.0f}"
                         for i in por_tramo.index], fontsize=7)
ax.set_title("Sesgo medio por tramo de gasto real\nEn el tramo alto se queda corto",
             fontweight="bold", fontsize=10, color="#922b21")
ax.set_xlabel("Tramo de gasto real (€)")
ax.set_ylabel("Sesgo medio (€)")
ax.grid(True, alpha=0.3, axis="y")

# 6. Por segmento, otra vez.
ax = axes[1, 2]
for i, segmento in enumerate(segmentos):
    dentro = (gasto["segmento"] == segmento).to_numpy()
    ax.scatter(predicho[dentro], residuo[dentro], s=13, alpha=0.5,
               edgecolors="none", color=colores[segmento], label=segmento)
ax.axhline(0, color="black", linewidth=1.6)
ax.set_title("El mismo residuo, por segmento", fontweight="bold", fontsize=10)
ax.set_xlabel("Gasto predicho (€)")
ax.set_ylabel("Residuo (€)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"Los dos problemas que el R² de {r2:.2f} no menciona:")
print()
print("1. EL EMBUDO. El segundo panel: la desviación del residuo pasa de "
      f"{desviaciones[0]:.0f} €")
print(f"   en el tramo bajo a {desviaciones[-1]:.0f} € en el alto, unas "
      f"{desviaciones[-1] / desviaciones[0]:.0f} veces más.")
print("   El modelo es fiable para los clientes de gasto pequeño e inútil para los")
print("   grandes, que son justo los que interesan.")
print()
print("   El tercer panel enseña la otra mitad: en términos RELATIVOS el error")
print("   es parecido en todo el rango. O sea que se equivoca un porcentaje")
print("   constante, y como los valores altos son grandes, el error absoluto se")
print("   dispara. Eso sugiere un arreglo concreto: modelar el logaritmo del gasto.")
print()
print(f"2. EL SESGO EN LA COLA. El quinto panel: el sesgo medio del tramo más alto es")
print(f"   {por_tramo['sesgo'].iloc[-1]:+.0f} €, mientras que en los demás está cerca "
      f"de cero.")
print("   No se equivoca al azar con los mejores clientes: se queda corto SIEMPRE.")
print("   Un error sistemático se puede corregir; uno al azar, no. Eso es una buena")
print("   noticia, pero solo si se ha visto.")
print()
print(f"Y el error medio global es {error_medio:+.1f} €, mientras que en el tramo alto")
print(f"es {por_tramo['sesgo'].iloc[-1]:+.0f} €: "
      f"{por_tramo['sesgo'].iloc[-1] / error_medio:.0f} veces mayor. La cifra global "
      f"no es que sea pequeña,")
print("es que promedia un sesgo enorme en una sexta parte de los clientes con un")
print("sesgo casi nulo en las otras cinco. Es exactamente la razón por la que hay que")
print("mirar el gráfico y no la cifra.")

## 10. Curvas de aprendizaje: si el problema es el modelo o los datos

El último gráfico no evalúa el resultado: **diagnostica qué hacer para mejorarlo**, y
es la diferencia entre pagar más datos o cambiar de modelo.

Se dibuja el error de entrenamiento y el de validación frente al número de ejemplos, y
se leen dos cosas:

| Lo que se ve | Se llama | Qué hacer |
|---|---|---|
| Hueco grande entre las dos curvas, y el de validación bajando | **Sobreajuste** | Más datos, o regularizar, o simplificar |
| Las dos curvas juntas y altas, planas | **Subajuste** | Más datos NO sirve. Modelo más potente, o mejores características |

La utilidad práctica es directa: **si las dos curvas están juntas y planas, conseguir
más datos es tirar el dinero**, y ese es un consejo de miles de euros.

In [ ]:
curvas_aprendizaje = pd.read_csv(os.path.join("datos", "curvas_aprendizaje.csv"))

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5)) 
fig.suptitle("Curvas de aprendizaje: el diagnóstico", fontsize=15, fontweight="bold")

titulos = {
    "sobreajuste": ("SOBREAJUSTE", "#c0392b",
                    "Hueco grande y validación bajando\n→ más datos SÍ ayudan"),
    "subajuste": ("SUBAJUSTE", "#e67e22",
                  "Curvas juntas, altas y planas\n→ más datos NO ayudan"),
}

for ax, modelo in zip(axes[:2], ["sobreajuste", "subajuste"]):
    datos = curvas_aprendizaje[curvas_aprendizaje["modelo"] == modelo]
    nombre, color, diagnostico = titulos[modelo]
    ax.plot(datos["n_entrenamiento"], datos["error_entrenamiento"], "o-",
            linewidth=2.4, markersize=7, color="#2471a3", label="Entrenamiento")
    ax.plot(datos["n_entrenamiento"], datos["error_validacion"], "s-",
            linewidth=2.4, markersize=7, color=color, label="Validación")
    ax.fill_between(datos["n_entrenamiento"], datos["error_entrenamiento"],
                    datos["error_validacion"], alpha=0.16, color=color,
                    label="El hueco")
    ax.set_xscale("log")
    ax.set_title(f"{nombre}\n{diagnostico}", fontweight="bold", fontsize=10,
                 color=color)
    ax.set_xlabel("Ejemplos de entrenamiento")
    ax.set_ylabel("Error")
    ax.set_ylim(0, 0.45)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, which="both")

# El tercer panel: el hueco de los dos, que es lo que hay que mirar.
ax = axes[2]
for modelo in ["sobreajuste", "subajuste"]:
    datos = curvas_aprendizaje[curvas_aprendizaje["modelo"] == modelo]
    hueco = datos["error_validacion"].to_numpy() - datos["error_entrenamiento"].to_numpy()
    ax.plot(datos["n_entrenamiento"], hueco, "o-", linewidth=2.4, markersize=7,
            color=titulos[modelo][1], label=f"{modelo}")
ax.axhline(0, color="black", linewidth=1.2)
ax.set_xscale("log")
ax.set_title("El hueco entre las dos curvas\nEs la cifra que resume el diagnóstico",
             fontweight="bold", fontsize=10)
ax.set_xlabel("Ejemplos de entrenamiento")
ax.set_ylabel("Error de validación − error de entrenamiento")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, which="both")

fig.tight_layout()
plt.show()

for modelo in ["sobreajuste", "subajuste"]:
    datos = curvas_aprendizaje[curvas_aprendizaje["modelo"] == modelo]
    primero, ultimo = datos.iloc[0], datos.iloc[-1]
    print(f"{modelo.upper()}")
    print(f"  con {int(primero['n_entrenamiento']):>5} ejemplos: "
          f"entrenamiento {primero['error_entrenamiento']:.3f}, "
          f"validación {primero['error_validacion']:.3f}, "
          f"hueco {primero['error_validacion'] - primero['error_entrenamiento']:.3f}")
    print(f"  con {int(ultimo['n_entrenamiento']):>5} ejemplos: "
          f"entrenamiento {ultimo['error_entrenamiento']:.3f}, "
          f"validación {ultimo['error_validacion']:.3f}, "
          f"hueco {ultimo['error_validacion'] - ultimo['error_entrenamiento']:.3f}")
    print()

sobre = curvas_aprendizaje[curvas_aprendizaje["modelo"] == "sobreajuste"]
sub = curvas_aprendizaje[curvas_aprendizaje["modelo"] == "subajuste"]

print("La decisión que sale de cada gráfico:")
print()
print(f"SOBREAJUSTE. El error de validación ha bajado de "
      f"{sobre['error_validacion'].iloc[0]:.2f} a "
      f"{sobre['error_validacion'].iloc[-1]:.2f} y sigue")
print(f"  bajando, y el hueco se ha cerrado de "
      f"{sobre['error_validacion'].iloc[0] - sobre['error_entrenamiento'].iloc[0]:.2f} a "
      f"{sobre['error_validacion'].iloc[-1] - sobre['error_entrenamiento'].iloc[-1]:.2f}. "
      f"Merece la pena conseguir")
print("  más datos: la curva todavía no se ha aplanado.")
print()
print(f"SUBAJUSTE. Las dos curvas están pegadas en torno a "
      f"{sub['error_validacion'].iloc[2:].mean():.2f} desde los "
      f"{int(sub['n_entrenamiento'].iloc[2])} ejemplos.")
print(f"  Pasar de {int(sub['n_entrenamiento'].iloc[2])} a "
      f"{int(sub['n_entrenamiento'].iloc[-1])} —"
      f"{sub['n_entrenamiento'].iloc[-1] / sub['n_entrenamiento'].iloc[2]:.0f} veces "
      f"más datos— no ha mejorado")
print("  nada. El problema no son los datos: es que el modelo no tiene capacidad para")
print("  representar la relación, o que las características no contienen la respuesta.")
print("  Comprar más datos aquí es gastar sin retorno.")

## 11. La lista de comprobación

Los seis gráficos que hay que mirar antes de creerse el resultado de un modelo. En
este orden, y ninguno sustituye a otro.

| | Gráfico | Detecta lo que ninguno de los demás detecta |
|---|---|---|
| 1 | **Reparto de puntuaciones por clase** | El techo del modelo. Si las dos clases se solapan, ningún umbral lo arregla |
| 2 | **Matriz de confusión, normalizada de las dos formas** | Qué tipo de error comete. Precisión y exhaustividad son dos preguntas |
| 3 | **Curva de precisión-exhaustividad** | El comportamiento en todos los umbrales, sin que la clase mayoritaria lo disimule |
| 4 | **Diagrama de fiabilidad** | Si la puntuación se puede leer como probabilidad. **Solo aquí** |
| 5 | **Todo lo anterior, por segmento** | El grupo en el que el modelo no funciona. Es también la comprobación de equidad |
| 6 | **Residuos frente a predicho** (regresión) o **curva de aprendizaje** | Estructura que queda en el error, y si más datos servirían |

### Y la regla que resume el cuaderno

> **Cada número resume, y cada resumen tiene un modo de fallar que consiste en que el
> resumen sale bien.** Por eso la evaluación de un modelo es un problema de
> visualización: los gráficos son lo que enseña la forma del fallo, y la forma del
> fallo es lo que decide qué se hace a continuación.

In [ ]:
# El cuadro de mando de evaluación completo: los seis en una figura, que es lo que
# hay que entregar en la P4.2.
fig = plt.figure(figsize=(16, 11))
gs = fig.add_gridspec(3, 3, hspace=0.48, wspace=0.3)

# 1. Reparto de puntuaciones.
ax = fig.add_subplot(gs[0, 0])
for clase, color, etiqueta in ((0, "#2471a3", "se queda"), (1, "#c0392b", "se va")):
    puntos = modelo_a[verdad == clase]
    ax.hist(puntos, bins=np.linspace(0, 1, 26),
            weights=np.ones(len(puntos)) / len(puntos),
            alpha=0.6, color=color, label=etiqueta)
ax.axvline(umbral_optimo, color="#1d6b3f", linestyle="--", linewidth=2)
ax.set_title("1. Reparto de puntuaciones", fontweight="bold", fontsize=10)
ax.set_xlabel("Puntuación", fontsize=8)
ax.set_ylabel("Proporción de su clase", fontsize=8)
ax.legend(fontsize=7)
ax.tick_params(labelsize=7)

# 2. Matriz de confusión.
ax = fig.add_subplot(gs[0, 1])
matriz_optima = pd.crosstab(abandono["abandona"],
                            (modelo_a >= umbral_optimo).astype(int)).to_numpy()
normalizada = matriz_optima / matriz_optima.sum(axis=1, keepdims=True)
ax.imshow(normalizada, cmap="Greens", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{matriz_optima[i, j]}\n{normalizada[i, j]:.0%}",
                ha="center", va="center", fontsize=10, fontweight="bold",
                color="white" if normalizada[i, j] > 0.55 else "black")
ax.set_title(f"2. Confusión (umbral {umbral_optimo:.2f})", fontweight="bold",
             fontsize=10)
ax.set_xticks([0, 1], ["dice 0", "dice 1"], fontsize=8)
ax.set_yticks([0, 1], ["es 0", "es 1"], fontsize=8)

# 3. Precisión-exhaustividad.
ax = fig.add_subplot(gs[0, 2])
ax.plot(curva_a["exhaustividad"], curva_a["precision"], linewidth=2.2,
        color="#1a5276")
ax.axhline(verdad.mean(), color="black", linestyle="--", linewidth=1.2)
ax.plot(tp_optimo, precision_optima, "*", color="#1d6b3f", markersize=15,
        markeredgecolor="white")
ax.set_title(f"3. Precisión-exhaustividad (AP {curva_a['ap']:.3f})",
             fontweight="bold", fontsize=10)
ax.set_xlabel("Exhaustividad", fontsize=8)
ax.set_ylabel("Precisión", fontsize=8)
ax.set_ylim(0, 1.02)
ax.tick_params(labelsize=7)
ax.grid(True, alpha=0.3)

# 4. Fiabilidad.
ax = fig.add_subplot(gs[1, 0])
ax.plot([0, 1], [0, 1], "k--", linewidth=1.4)
ax.plot(fiab_a[:, 0], fiab_a[:, 1], "o-", linewidth=2.2, markersize=7,
        color="#1a5276", label="Modelo A")
ax.plot(fiab_b[:, 0], fiab_b[:, 1], "s-", linewidth=2.2, markersize=7,
        color="#c0392b", label="Modelo B")
ax.set_title("4. Fiabilidad (calibración)", fontweight="bold", fontsize=10)
ax.set_xlabel("Lo que dice el modelo", fontsize=8)
ax.set_ylabel("Lo que ocurre", fontsize=8)
ax.legend(fontsize=7)
ax.tick_params(labelsize=7)
ax.grid(True, alpha=0.3)

# 5. Por segmento.
ax = fig.add_subplot(gs[1, 1])
ax.bar(range(len(segmentos)), resumen["auc"],
       color=[colores[s] for s in segmentos], edgecolor="black", linewidth=0.6)
ax.axhline(curva_a["auc"], color="black", linestyle="--", linewidth=1.6,
           label=f"global {curva_a['auc']:.3f}")
ax.axhline(0.5, color="0.5", linestyle=":", linewidth=1.4, label="azar")
ax.set_xticks(range(len(segmentos)), segmentos, fontsize=8)
ax.set_ylim(0.4, 1)
ax.set_ylabel("AUC", fontsize=8)
ax.set_title("5. AUC por segmento", fontweight="bold", fontsize=10)
ax.legend(fontsize=7)
ax.tick_params(labelsize=7)

# 6. Coste según el umbral.
ax = fig.add_subplot(gs[1, 2])
ax.plot(np.linspace(0.01, 0.99, 300), coste_total, linewidth=2.2, color="#1a5276")
ax.axvline(umbral_optimo, color="#1d6b3f", linestyle="--", linewidth=1.8)
ax.axvline(0.5, color="#c0392b", linestyle=":", linewidth=1.8)
ax.set_title("6. Coste según el umbral", fontweight="bold", fontsize=10)
ax.set_xlabel("Umbral", fontsize=8)
ax.set_ylabel("Coste (€)", fontsize=8)
ax.tick_params(labelsize=7)
ax.grid(True, alpha=0.3)

# La fila de abajo: los residuos y la curva de aprendizaje.
ax = fig.add_subplot(gs[2, 0:2])
ax.scatter(predicho, residuo, s=11, alpha=0.45, edgecolors="none", color="#c0392b")
ax.axhline(0, color="black", linewidth=1.4)
ax.plot(centros, desviaciones, color="#1d6b3f", linewidth=2)
ax.plot(centros, -desviaciones, color="#1d6b3f", linewidth=2)
ax.set_title("7. Residuos del modelo de regresión (embudo y sesgo en la cola)",
             fontweight="bold", fontsize=10)
ax.set_xlabel("Gasto predicho (€)", fontsize=8)
ax.set_ylabel("Residuo (€)", fontsize=8)
ax.tick_params(labelsize=7)
ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[2, 2])
for modelo in ["sobreajuste", "subajuste"]:
    datos = curvas_aprendizaje[curvas_aprendizaje["modelo"] == modelo]
    ax.plot(datos["n_entrenamiento"], datos["error_validacion"], "o-",
            linewidth=2, markersize=5, color=titulos[modelo][1], label=modelo)
    ax.plot(datos["n_entrenamiento"], datos["error_entrenamiento"], "--",
            linewidth=1.4, color=titulos[modelo][1], alpha=0.6)
ax.set_xscale("log")
ax.set_title("8. Curvas de aprendizaje", fontweight="bold", fontsize=10)
ax.set_xlabel("Ejemplos", fontsize=8)
ax.set_ylabel("Error", fontsize=8)
ax.legend(fontsize=7)
ax.tick_params(labelsize=7)
ax.grid(True, alpha=0.3, which="both")

fig.suptitle("Cuadro de mando de evaluación · modelo de abandono de TechStore",
             fontsize=16, fontweight="bold")
plt.show()

print("Este cuadro de mando es el entregable central de la práctica P4.2, y lo que")
print("hay que saber defender es POR QUÉ está cada panel: qué detecta que ninguno de")
print("los otros detecta.")

## Ejercicios

### Ejercicio 1. El punto de trabajo con otros costes

La sección 5 usa 20 € de falso positivo y 180 € de falso negativo. Repítela con estos
tres escenarios y rellena la tabla:

| Escenario | Falso positivo | Falso negativo |
|---|---|---|
| Correo automático de retención | 0,50 € | 180 € |
| Llamada de un comercial | 45 € | 180 € |
| Regalo de un producto | 150 € | 180 € |

Para cada uno: el umbral óptimo, cuántos clientes se marcan, la precisión, la
exhaustividad y el coste. Y contesta: **¿en cuál de los tres deja de merecer la pena
usar el modelo?** Compáralo con las dos estrategias tontas (no hacer nada y avisar a
todos).

### Ejercicio 2. Cuatro matrices con el mismo F1

La sección 2 construye cuatro matrices con la misma exactitud. Haz lo mismo con el
**F1**: cuatro matrices de confusión sobre 400 casos con 96 positivos, con F1 idéntico
a dos decimales y comportamientos distintos.

Dibújalas y explica qué esconde el F1 que no escondía la exactitud, y al revés.

### Ejercicio 3. La ROC no ve la calibración

Construye **tú** un tercer modelo, el C, que tenga la misma curva ROC que A y B y una
calibración distinta de las dos: que esté **infraconfiado**, o sea que diga menos de
lo que ocurre.

1. Encuentra la transformación (pista: la de B es `p ** 0.42`).
2. Comprueba que el AUC coincide con el de A hasta el cuarto decimal.
3. Dibuja los tres en el diagrama de fiabilidad.
4. Explica en qué decisión de negocio un modelo infraconfiado hace daño, y en cuál no
   importa.

### Ejercicio 4. Buscar el segmento malo sin que te lo digan

En la sección 8 se sabía de antemano que había que mirar `segmento`. En la vida real no
te lo dicen.

Escribe una función `busca_grupos_malos(datos, verdad, puntuacion, columnas)` que:

1. Para cada columna categórica de la lista, y para cada tramo de las numéricas
   (divídelas en tres con `pd.qcut`), calcule el AUC del grupo.
2. Devuelva un `DataFrame` ordenado por AUC, con el tamaño de cada grupo.
3. **Descarte los grupos con menos de 30 casos**, porque su AUC es ruido. Explica por
   qué ese filtro es imprescindible y no una precaución.

Pásasela a las cuatro columnas numéricas de `abandono` y a `segmento`, y comprueba si
encuentra algún grupo problemático que la sección 8 no mencionó.

### Ejercicio 5. Arreglar el modelo de regresión

La sección 9 encuentra dos defectos: el error crece con el valor, y hay un sesgo
negativo en la cola alta.

1. Dibuja el residuo del **logaritmo**: `log(predicho) − log(real)`. ¿Desaparece el
   embudo? Explica por qué eso apunta a modelar el logaritmo del gasto.
2. Corrige el sesgo de la cola: calcula el factor por el que habría que multiplicar
   las predicciones del decil superior para que su sesgo medio sea cero, aplícalo, y
   vuelve a dibujar los seis paneles.
3. Recalcula R², MAE y RMSE después de la corrección. **¿Mejoran los tres?** Si alguno
   empeora, explica por qué, y qué dice eso de usar una sola cifra.

### Ejercicio 6. El informe de una página

Con todo lo anterior, escribe **una página** dirigida a la dirección de TechStore, que
no programa, contestando a una sola pregunta: *¿usamos este modelo o no?*

Tiene que llevar:

1. La recomendación, en la primera línea.
2. **Una sola figura**, la que mejor la sostenga. Elígela y justifica la elección.
3. La cifra de negocio: cuánto se ahorra al año con el punto de trabajo que propones.
4. Los tres límites del modelo, con el mismo cuidado que los aciertos. Uno de ellos
   tiene que ser el segmento en el que no funciona.
5. Qué haría falta para mejorarlo, citando la curva de aprendizaje.

La restricción es la que importa: **una sola figura**. Elegir cuál obliga a decidir
qué es lo principal, y es exactamente lo que se evalúa en la P4.2.

## Lo que hay que llevarse de aquí

1. **La exactitud casi nunca sirve.** Con clases desequilibradas, la regla tonta ya
   saca un número alto, y cuanto más desequilibradas, más alto.
2. **Cuatro modelos pueden tener la misma exactitud** y comportamientos que no se
   parecen en nada. El resumen no distingue lo que hay que decidir.
3. **La matriz de confusión se normaliza de dos formas**, y son dos preguntas
   distintas que le importan a personas distintas.
4. **El reparto de puntuaciones por clase enseña el techo del modelo.** Si se solapan,
   ningún umbral lo arregla.
5. **El umbral sale del coste de cada error**, no de 0,5. Elegirlo bien vale dinero
   sin reentrenar nada.
6. **Una curva ROC es un `argsort` y dos `cumsum`.** Construirla una vez a mano quita
   la caja negra para siempre.
7. **Con la clase de interés en minoría, la curva que decide es la de
   precisión-exhaustividad**, y su línea de azar es la prevalencia, no una diagonal.
8. **La ROC solo depende del orden y no ve la calibración.** Dos modelos con la misma
   curva pueden decir 0,50 y 0,14 sobre lo mismo, y solo uno se puede usar para
   decidir con euros.
9. **Todo se desglosa por segmento.** Un modelo que funciona de media y falla en un
   grupo es un problema práctico y, si el grupo son personas, un problema de equidad.
10. **En regresión, el gráfico es el residuo frente al predicho**, y lo que se busca
    es estructura: embudo, curva o desplazamiento.
11. **La curva de aprendizaje dice si más datos servirían.** Dos curvas juntas y
    planas significan que conseguir más datos es gastar sin retorno.

Esto es lo que el criterio 2.e llama *evaluar los resultados obtenidos*. En la UD5 se
entrenarán modelos y se llamará a `sklearn.metrics`, que calcula todo esto en una
línea. La diferencia será que ya se sabe qué hace por dentro y, sobre todo, **qué
pregunta contesta cada gráfico**.